<a href="https://colab.research.google.com/github/JakeFRCSE/geometry-of-truth-reproduction/blob/main/notebooks/PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0. Loading Data

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
class DataFrameDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            "statement" : row["statement"],
            "label" : row["label"],
        }

def load_dataset(name):
    return pd.read_csv(DATASET_DIR / name)

datasets = {
    name.split(".")[0] : load_dataset(name)
    for name in dataset_names
}

datasets_torch = {
    k: DataFrameDataset(v)
    for k, v in datasets.items()
}

dataloaders = {
    k: DataLoader(v, batch_size=16, shuffle=False)
    for k, v in datasets_torch.items()
}

cities_loader = dataloaders["cities"]
neg_cities_loader = dataloaders["neg_cities"]

In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/research/research-cycle1/throw-away-projects/geometry-of-truth")
DATASET_DIR = BASE_DIR / "dataset"
dataset_names = ["cities.csv", "neg_cities.csv"]

ACTS_DIR = BASE_DIR / "dataset" / MODEL_NAME
RESULTS_DIR = BASE_DIR / "results" / MODEL_NAME

ACTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEBUG = False
if DEBUG:
    tracer_kwargs = {"scan" : True, "validate" : False}
else:
    tracer_kwargs = {"scan" : True, "validate" : False}

# 1. Loading Model

In [ ]:
! pip install nnsight

In [ ]:
from pathlib import Path
import torch as t
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from nnsight import LanguageModel

In [ ]:
MODEL_NAME = "google/gemma-2b-it"

t.set_grad_enabled(False)

model = LanguageModel(
    MODEL_NAME,
    device_map="auto",
    dtype=t.bfloat16
)

tokenizer = model.tokenizer

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. Defining Functions

In [ ]:
def unwrap_model(model):
    if hasattr(model, "model"):
        return model.model
    return model

def get_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    if hasattr(model, "gpt_neox") and hasattr(model.gpt_neox, "layers"):
        return model.gpt_neox.layers
    raise AttributeError("Could not find transformer layers.")


def get_final_norm(model):
    if hasattr(model, "model") and hasattr(model.model, "norm"):
        return model.model.norm
    if hasattr(model, "transformer") and hasattr(model.transformer, "ln_f"):
        return model.transformer.ln_f
    if hasattr(model, "gpt_neox") and hasattr(model.gpt_neox, "final_layer_norm"):
        return model.gpt_neox.final_layer_norm
    return None

def get_lm_head(model):
    m = unwrap_model(model)

    if hasattr(m, "lm_head"):
        return m.lm_head

    if hasattr(m, "embed_out"):
        return m.embed_out

def get_layer_output(layer):
    out = layer.output
    if isinstance(out, tuple):
        out = out[0]
    return out

def get_logits_from_hidden(model, hidden):
    final_norm = get_final_norm(model)
    if final_norm is not None:
        hidden = final_norm(hidden)
    lm_head = get_lm_head(model)
    return lm_head(hidden)

def collect_activations(dataloader, layer_idx):
    all_acts = []
    all_labels = []

    for batch in tqdm(dataloader):
        texts = list(batch["statement"])
        labels = t.as_tensor(batch["label"], dtype=t.long)

        with model.trace(texts, **tracer_kwargs):
            acts = get_layer_output(layers[layer_idx])[:, -1, :].save()

        all_acts.append(acts.float().cpu())
        all_labels.append(labels.cpu())

    return t.cat(all_acts, dim=0), t.cat(all_labels, dim=0)

def compute_pca(x):
    x = x.float()
    x = x - x.mean(dim=0, keepdim=True)
    _, _, vh = t.linalg.svd(x, full_matrices=False)
    return x @ vh[:2].T

def save_pca_plot(proj, labels, layer_idx, name):
    plot_dir = RESULTS_DIR / name
    plot_dir.mkdir(parents=True, exist_ok=True)

    plot_path = plot_dir / f"layer_{layer_idx:02d}.png"

    proj_np = proj.cpu().numpy()
    labels_np = labels.cpu().numpy()

    plt.figure(figsize=(5, 5))
    plt.scatter(
        proj_np[labels_np == 0, 0],
        proj_np[labels_np == 0, 1],
        s=10,
        alpha=0.7,
        color="red",
        label="False",
    )
    plt.scatter(
        proj_np[labels_np == 1, 0],
        proj_np[labels_np == 1, 1],
        s=10,
        alpha=0.7,
        color="blue",
        label="True",
    )
    plt.legend()
    model_name_short = MODEL_NAME.split("/")[-1]
    plt.title(f"{name} | {model_name_short} | Layer {layer_idx}")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.tight_layout()
    plt.savefig(plot_path, dpi=200, bbox_inches="tight")
    plt.close()

    print(f"Saved plot: {plot_path}")

def run(layer_idx, name, dataloader):
    acts, labels = collect_activations(dataloader, layer_idx)
    proj = compute_pca(acts)

    save_path = ACTS_DIR / name / f"layer_{layer_idx:02d}.pt"
    save_path.parent.mkdir(parents=True, exist_ok=True)

    t.save(
        {
            "acts": acts,
            "proj": proj,
            "labels": labels,
        },
        save_path,
    )

    print(f"Saved tensor: {save_path}")

    save_pca_plot(proj, labels, layer_idx, name)

# 4. Run

In [ ]:
layers = get_layers(model)

NUM_LAYERS = len(layers)

for layer_idx in range(NUM_LAYERS):
    run(layer_idx, "cities", cities_loader)